# Neural Network Fundamentals

> 📘 **Python Mastery** · Module 14 — Deep Learning · Lesson 1/7

Deep learning is just linear algebra plus a smart way to tune numbers — and today you build every core piece (neuron, activation, loss, gradient descent, backprop) yourself in NumPy.

## 🎯 Learning Objectives

- **Map** the parts of a biological neuron onto the mathematics of an artificial neuron.
- **Compute** a perceptron's weighted sum, bias, and activation with NumPy's `dot` product.
- **Implement** Sigmoid, Tanh, and ReLU activations and explain their trade-offs (including the vanishing-gradient problem).
- **Run** a forward pass through a 2-4-1 network using matrix notation `X @ W + b`.
- **Compute** Mean Squared Error and Binary Cross-Entropy losses and know when each applies.
- **Explain** gradient descent, learning-rate sensitivity, and the chain-rule intuition behind backpropagation.

## 1. From Biology to Mathematics: the Artificial Neuron

Your brain has ~86 billion neurons, each receiving electrical pulses through branch-like **dendrites**, combining them in the **cell body**, and — if the combined signal beats a threshold — firing a pulse down its **axon**. In 1943 McCulloch and Pitts realised you can write that idea with nothing but multiplication and addition.

| Biological neuron | Artificial neuron | Job |
|---|---|---|
| Dendrites (inputs) | Features $x_1, x_2, \dots, x_n$ | Receive signals |
| Synapse strength | Weights $w_1, w_2, \dots, w_n$ | How much each input matters |
| Cell body (integration) | Weighted sum $z = \sum_i w_i x_i + b$ | Add up evidence |
| Firing threshold | Bias $b$ + activation $f(z)$ | Decide whether (and how strongly) to fire |
| Axon (output) | Output $a = f(z)$ | Signal passed to the next neuron |

The **bias** is just a weight attached to a constant input of 1 — it lets the neuron fire even when all inputs are weak, shifting the decision threshold left or right.

**Syntax:**
```python
import numpy as np

x = np.array([...])          # inputs
w = np.array([...])          # weights
b = 0.5                      # bias
z = np.dot(w, x) + b         # weighted sum  (w · x + b)
a = activation(z)            # non-linear "fire" decision
```

In [ ]:
import numpy as np

# Scenario: should Sarah go for an evening run in Dhaka?
# inputs: [temperature comfort (0-10), energy level (0-10), free time (0-10)]
w = np.array([0.6, 0.9, 0.4])     # learned weights: energy matters most
b = -3.0                          # needs a minimum total push to fire

def step(z):
    """Perceptron activation: fire (1) only above threshold."""
    return 1 if z > 0 else 0

evening = np.array([7.0, 8.0, 6.0])   # lovely evening
rainy   = np.array([3.0, 2.0, 5.0])   # tired, muggy day

for name, x in [("Lovely evening", evening), ("Rainy tired day", rainy)]:
    z = np.dot(w, x) + b              # one number: total evidence
    print(f"{name:18s} z = {z:5.2f} -> go running? {bool(step(z))}")

# The same computation, fully vectorized for several days at once:
X = np.array([evening, rainy])        # shape (2 days, 3 features)
Z = X @ w + b                         # shape (2,) -- one score per day
print("vectorized scores:", Z.round(2))

## 2. The Perceptron (and Why One Neuron Is Not Enough)

A **perceptron** is a single artificial neuron with a hard threshold. Its decision boundary is always a straight line (or flat plane) — it can only separate data that is *linearly separable*. The classic proof of failure is **XOR**: no single straight line can put `(0,1)` and `(1,0)` on one side and `(0,0)` and `(1,1)` on the other.

The fix is two-fold: **stack neurons in layers** (so later neurons can combine earlier lines) and make activations **non-linear** (so stacked layers can bend the boundary). That combination is literally a neural network.

**Syntax:**
```python
Z = X @ W + b        # X: (n_samples, n_features), W: (n_features, 1)
P = sigmoid(Z)       # squashes scores into (0, 1) -> probability
```

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Can a single neuron solve XOR? Inputs and targets:
X  = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y  = np.array([0, 1, 1, 0])           # XOR truth table

best_acc = 0
for w1 in (-1, 1, 2):
    for w2 in (-1, 1, 2):
        for b in (-1.5, -0.5, 0.5, 1.5):
            p = (sigmoid(X @ np.array([w1, w2]) + b) > 0.5).astype(int)
            best_acc = max(best_acc, (p == y).mean())

print(f"Best XOR accuracy over 36 single neurons: {best_acc:.0%}")
print("-> impossible to exceed 75%: one line cannot carve XOR.")
print("-> we need LAYERS of neurons with non-linear activations.")

## 3. Activation Functions: Where the Non-Linearity Lives

Without activation functions, ten stacked layers collapse into one big linear map — no more powerful than a single layer. Activations are the cheap non-linear "bend" applied after every weighted sum.

| Function | Formula | Output range | Typical use | Watch out |
|---|---|---|---|---|
| Sigmoid $\sigma(z)$ | $1/(1+e^{-z})$ | $(0,1)$ | Output layer for binary probability | **Vanishing gradient** — derivative peaks at just 0.25 and dies for large \|z\| |
| Tanh | $(e^z-e^{-z})/(e^z+e^{-z})$ | $(-1,1)$ | Classic hidden layers, RNN cells | Still saturates; same vanishing issue |
| ReLU | $\max(0, z)$ | $[0,\infty)$ | Default choice for hidden layers | "Dead" neurons stuck at 0 if $z$ always negative |
| LeakyReLU | $\max(0.01z,\ z)$ | $(-\infty,\infty)$ | Fix for dying ReLUs | Adds one slope hyper-parameter |

**Syntax:**
```python
def sigmoid(z): return 1 / (1 + np.exp(-z))
def tanh(z):    return np.tanh(z)
def relu(z):    return np.maximum(0, z)
def leaky_relu(z, alpha=0.01): return np.where(z > 0, z, alpha * z)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z): return 1 / (1 + np.exp(-z))
def d_sigmoid(z): return sigmoid(z) * (1 - sigmoid(z))   # sigma * (1 - sigma)

funcs  = [("Sigmoid", sigmoid, d_sigmoid),
          ("Tanh",    np.tanh, lambda z: 1 - np.tanh(z) ** 2),
          ("ReLU",    lambda z: np.maximum(0, z), lambda z: (z > 0).astype(float))]

z = np.linspace(-6, 6, 300)
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)

for col, (name, f, df) in enumerate(funcs):
    axes[0, col].plot(z, f(z), lw=2)
    axes[0, col].set_title(name); axes[0, col].axhline(0, color="gray", lw=0.5)
    axes[0, col].grid(alpha=0.3)
    axes[1, col].plot(z, df(z), lw=2, color="tab:red")
    axes[1, col].set_title(f"{name} derivative"); axes[1, col].axhline(0, color="gray", lw=0.5)
    axes[1, col].grid(alpha=0.3)

axes[0, 0].set_ylabel("activation")
axes[1, 0].set_ylabel("derivative")
fig.suptitle("Activation functions and their derivatives (top: value, bottom: slope)", y=1.02)
plt.tight_layout()
plt.show()

print("Max sigmoid derivative:", round(d_sigmoid(0.0), 2),
      "-> deep stacks multiply many factors <= 0.25: gradients vanish.")
print("ReLU derivative is 1 for every positive z -> gradient flows untouched.")

> 🔍 **Under the Hood:** `np.exp` on an array runs a compiled C loop over a contiguous buffer — roughly 100x faster than a Python `for`. That is why we express networks as whole-array operations (`X @ W`) instead of per-neuron loops: the maths is identical, but the CPU/GPU does one vectorised sweep instead of millions of interpreted steps. Every deep-learning framework is built on this same trick.

## 4. Forward Pass of a 2-4-1 Network

A network with 2 inputs, 4 hidden neurons, and 1 output is just two matrix multiplications in sequence. Each **column** of `W1` is one hidden neuron's complete set of weights, so one `@` computes all four neurons for every sample at once.

$$Z_1 = X W_1 + b_1, \quad A_1 = \mathrm{ReLU}(Z_1), \quad Z_2 = A_1 W_2 + b_2$$

**Syntax:**
```python
Z1 = X  @ W1 + b1     # X: (N, 2) -> Z1: (N, 4)   hidden pre-activations
A1 = np.maximum(Z1, 0)                    # ReLU, element-wise
Z2 = A1 @ W2 + b2     # (N, 4) @ (4, 1) -> (N, 1) output scores (logits)
```

In [ ]:
import numpy as np

rng = np.random.default_rng(42)               # seeded -> reproducible

# Shapes: X(N,2) @ W1(2,4) + b1(1,4)  ->  A1(N,4) @ W2(4,1) + b2(1,1)
W1 = rng.normal(0, 0.5, size=(2, 4)); b1 = np.zeros((1, 4))
W2 = rng.normal(0, 0.5, size=(4, 1)); b2 = np.zeros((1, 1))

X = np.array([[0.5, 1.5],      # 4 samples, 2 features each
              [2.0, 0.5],
              [1.0, 1.0],
              [3.0, 2.5]])

Z1 = X @ W1 + b1               # (4, 2) @ (2, 4) -> (4, 4)
print("Z1 shape:", Z1.shape)
A1 = np.maximum(Z1, 0)         # ReLU element-wise
print("A1 shape:", A1.shape)
Z2 = A1 @ W2 + b2              # (4, 4) @ (4, 1) -> (4, 1)
print("Z2 shape:", Z2.shape)

print("\nscores (logits):", Z2.ravel().round(3))
print("probabilities :", (1 / (1 + np.exp(-Z2))).ravel().round(3))

# b1 has shape (1, 4) but X @ W1 has shape (4, 4):
# NumPy BROADCASTS the single bias row down all 4 samples.

## 5. Loss Functions: Putting a Number on "Wrong"

The **loss** converts all your model's errors into one number that training tries to shrink. Pick it by task type:

| Loss | Formula | Use when | Punishes |
|---|---|---|---|
| Mean Squared Error (MSE) | $\frac{1}{N}\sum (y-\hat y)^2$ | Regression (price, temperature) | Large errors brutally (squared) |
| Binary Cross-Entropy (BCE) | $-\frac{1}{N}\sum \big[y\log p + (1-y)\log(1-p)\big]$ | Binary classification with $p \in (0,1)$ | **Confident mistakes** — predicting 0.01 when the truth is 1 costs $-\log(0.01)\approx 4.6$ |

Using MSE with a sigmoid output is a classic beginner trap: when the sigmoid saturates its gradient nearly disappears, so the model learns at a crawl. Cross-entropy keeps gradients healthy even there.

**Syntax:**
```python
mse  = np.mean((y_true - y_pred) ** 2)
bce  = -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
```

In [ ]:
import numpy as np

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def bce(y_true, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)             # guard log(0)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

y = np.array([1.0, 0.0, 1.0])

# --- regression example: predicted Dhaka rent vs truth (in 1000s BDT)
truth    = np.array([20.0, 35.0])
pred_good = np.array([21.0, 34.0])
pred_bad  = np.array([30.0, 45.0])
print(f"MSE good guess: {mse(truth, pred_good):.2f}   bad guess: {mse(truth, pred_bad):.2f}")

# --- classification example: spam detector confidence
p_close  = np.array([0.9, 0.1, 0.8])   # close to truth [1, 0, 1]
p_confident_wrong = np.array([0.01, 0.99, 0.8])
print(f"BCE close: {bce(y, p_close):.3f}   confidently wrong: {bce(y, p_confident_wrong):.3f}")
print("-> BCE explodes for confident mistakes: exactly the pressure we want.")

## 6. Gradient Descent: Learning by Walking Downhill

Training = minimising the loss. Imagine standing fog-covered on a hillside (the loss surface): feel the slope under your feet (the **gradient**), step downhill, repeat. The **learning rate** is your stride length.

$$x_{new} = x - \text{lr} \cdot f'(x)$$

**Syntax:**
```python
for step in range(n_steps):
    grad = gradient(x)          # slope at current point
    x -= lr * grad              # one downhill step
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(x):  return (x - 3) ** 2        # bowl with minimum at x = 3
def g(x):  return 2 * (x - 3)         # derivative: slope of the bowl

lr, x = 0.3, 0.0                      # stride length, starting point
path = [x]

print("step |     x      |   f(x)")
for step in range(1, 26):
    x -= lr * g(x)                    # <-- THE learning rule
    path.append(x)
    if step % 5 == 0:
        print(f"{step:4d} | {x:10.5f} | {f(x):.2e}")

xs = np.linspace(-1.5, 7.5, 200)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(xs, f(xs), lw=2, label="f(x) = (x-3)^2")
sc = ax.scatter(path, f(np.array(path)), c=range(len(path)), cmap="viridis", s=45, zorder=3)
fig.colorbar(sc, ax=ax, label="iteration")
ax.annotate("start", (path[0], f(path[0])), xytext=(-1.2, 12), fontsize=9)
ax.annotate("minimum", (3, 0), xytext=(4.2, 3), fontsize=9)
ax.set_xlabel("x"); ax.set_ylabel("f(x)")
ax.set_title(f"Gradient descent trajectory (lr={lr})")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

### 6.1 Learning-Rate Sensitivity: the Goldilocks Parameter

Same bowl, three strides: too small crawls forever, too large overshoots and *diverges*, right-sized glides to the bottom. For this quadratic the maths says convergence needs $\text{lr} < 1$ — at $\text{lr} = 1.1$ each step multiplies the error by $-1.2$, so it grows 20% every iteration.

In [ ]:
import numpy as np

def g(x): return 2 * (x - 3)          # gradient of (x-3)^2

for lr in [0.01, 0.9, 1.1]:
    x, xs = 0.0, []
    for _ in range(15):
        x -= lr * g(x)
        xs.append(abs(x - 3))         # distance from the true minimum
    status = ("too slow" if lr == 0.01 else
              "converged!" if xs[-1] < 1e-6 else "DIVERGED")
    print(f"lr={lr:<5} |x-3| after 15 steps = {xs[-1]:.4f}  ({status})")

# Watch the oscillation at lr = 0.9: it zig-zags left/right but shrinks.
x = 0.0
trail = []
for _ in range(6):
    x -= 0.9 * g(x)
    trail.append(round(x, 3))
print("lr=0.9 zig-zag:", trail)

## 7. Backpropagation: the Chain Rule, Run Backwards

Training needs $\partial L/\partial w$ for *every* weight, but weights live buried inside nested functions. The **chain rule** un-nests them: to know how the final loss responds to an early weight, multiply together the local sensitivities of every step between them.

Think of a factory assembly line: if the finished product is flawed, you trace blame *backwards station by station*, asking each one "how much did my output change when my input changed a little?" Backpropagation is exactly that audit, done with derivatives and reused so nothing is computed twice.

Follow one tiny graph by hand: $x=1,\ b=0.5,\ w=2$, then $z = wx+b$, $a = \sigma(z)$, $L=(a-t)^2$ with target $t=1$.

**Syntax:**
```python
dL_da = 2 * (a - t)          # local: loss wrt its own input
da_dz = a * (1 - a)          # local: sigmoid wrt z
dz_dw = x                    # local: z = w*x + b wrt w
dL_dw = dL_da * da_dz * dz_dw   # chain rule: multiply along the path
```

In [ ]:
import numpy as np

# ---------- FORWARD: store every intermediate ----------
x, b, t = 1.0, 0.5, 1.0
w = 2.0

z = w * x + b                     # weighted sum
a = 1 / (1 + np.exp(-z))          # sigmoid
L = (a - t) ** 2                  # squared loss
print(f"forward : z={z:.3f}  a={a:.4f}  L={L:.6f}")

# ---------- BACKWARD: chain rule, one link at a time ----------
dL_da = 2 * (a - t)               # d/d a of (a-t)^2
da_dz = a * (1 - a)               # sigmoid'
dz_dw = x                         # d/d w of (w*x + b)
dz_db = 1.0                       # d/d b of (w*x + b)

dL_dw = dL_da * da_dz * dz_dw
dL_db = dL_da * da_dz * dz_db
print(f"backward: dL/da={dL_da:+.5f}  da/dz={da_dz:.5f}")
print(f"          dL/dw={dL_dw:+.6f}  dL/db={dL_db:+.6f}")

# ---------- PROOF: numerical gradient must agree ----------
eps = 1e-6
def loss_with(weight):
    zz = weight * x + b
    aa = 1 / (1 + np.exp(-zz))
    return (aa - t) ** 2
num_dw = (loss_with(w + eps) - loss_with(w - eps)) / (2 * eps)
print(f"numerical dL/dw = {num_dw:+.6f}  (analytic matches!)")

> 🔍 **Under the Hood:** real frameworks do precisely what you just did by hand — they record every operation in a **computational graph** during the forward pass, then replay it in reverse multiplying local derivatives. PyTorch's `loss.backward()` walks this graph with dynamic programming, so shared sub-expressions (like `da_dz` feeding both `dL_dw` and `dL_db`) are computed once. You built a 3-node engine; theirs handles billions of nodes with the same chain rule.

**PyTorch version** (requires `pip install torch`)

Everything above translates line-for-line into PyTorch. `nn.Linear(in, out)` creates both the weight matrix and the bias with correct shapes — no shape bookkeeping:

```python
import torch
import torch.nn as nn

layer1 = nn.Linear(2, 4)   # holds W1 (4x2) and b1 (4)
layer2 = nn.Linear(4, 1)   # holds W2 (1x4) and b2 (1)

X  = torch.tensor([[0.5, 1.5], [2.0, 0.5], [1.0, 1.0], [3.0, 2.5]])
Z1 = layer1(X)             # X @ W1.T + b1
A1 = torch.relu(Z1)        # same ReLU you just wrote in NumPy
Z2 = layer2(A1)            # output scores, shape (4, 1)
print(torch.sigmoid(Z2))   # probabilities in (0, 1)
```

From the next lesson on, tensors like these are our everyday containers.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Using MSE + sigmoid for classification | Sigmoid saturation makes gradients vanish; learning stalls | Use Binary Cross-Entropy on the output probability (or logits) |
| Initialising all weights to zero | Every neuron computes identically and receives identical gradients — the network never differentiates | Small random init, e.g. `rng.normal(0, np.sqrt(2/fan_in), shape)` (He init) |
| Learning rate too high | Loss shoots up, becomes `nan` | Drop lr by 10x until loss falls smoothly; consider schedules (Lesson 4) |
| Putting ReLU on the final output | Outputs can never be negative probabilities or prices | Match output activation to task: sigmoid for binary, none/linear for regression |
| Forgetting the bias term | Model forced through the origin; fits far fewer patterns | Always include `b` (frameworks add it by default with `bias=True`) |
| Reading accuracy as the training signal | Accuracy is flat/discrete — useless as a gradient target | Optimise a smooth **loss**; report accuracy only as a metric |

## 💡 Best Practices & Pro Tips

- **Vectorise everything.** Replace per-sample loops with `X @ W`; the maths is identical and it is 100x faster — the single habit that separates smooth ML code from painful ML code.
- **Seed every random draw** (`np.random.default_rng(seed)`): reproducible bugs are findable bugs.
- **Standardise inputs** to roughly zero-mean/unit-variance before training; unscaled features make the loss surface a narrow canyon that gradient descent struggles to descend.
- **Watch the loss curve first**, metrics second — the loss tells you *how* training is failing (too high lr = spikes; too low = flat line).
- **Start tiny.** A 2-4-1 network that trains in milliseconds is the right place to debug shapes and gradients before scaling up.
- **AI-engineering relevance:** every framework you will ever touch (`torch.nn`, Keras, JAX) compiles down to exactly these primitives — weighted sums, activations, a loss, and gradients. When debugging production models you will fall back on this mental model constantly.

## 📌 Summary

| Concept | What it does | Example |
|---|---|---|
| Weighted sum | Combines inputs with importance weights | `z = np.dot(w, x) + b` |
| Sigmoid | Score → probability in (0, 1) | `1 / (1 + np.exp(-z))` |
| ReLU | Keeps positives, zeroes negatives | `np.maximum(0, z)` |
| Forward pass | Layer-by-layer matrix pipeline | `Z1 = X @ W1 + b1; A1 = np.maximum(Z1, 0)` |
| MSE | Regression loss | `np.mean((y - y_hat)**2)` |
| BCE | Classification loss | `-np.mean(y*np.log(p) + (1-y)*np.log(1-p))` |
| Gradient step | Move parameters downhill | `x -= lr * grad` |

Key takeaways:
- A neural network = stacked weighted sums separated by non-linear activations; depth lets simple units compose complex boundaries.
- The loss is the single number training minimises; choose it by task (MSE ↔ regression, BCE ↔ classification).
- Gradient descent repeats *measure slope → step downhill*; the learning rate controls whether that converges, crawls, or explodes.
- Backpropagation is just the chain rule applied backwards through the forward computations — frameworks automate it, but you now know exactly what they automate.

## 🔗 Next Lesson

**02_Tensors_And_Autograd** — the same maths, upgraded to framework-grade containers (tensors) with automatic differentiation (autograd) doing your backprop for you.